# Course 2 — Linear Regression II

Multiple regression, diagnostics, qualitative predictors and
interactions. We continue with `Boston` and bring in `Carseats`.

**Sections**
1. Multiple LR and collinearity (0:00–0:30)
2. Diagnostic plots and leverage (0:30–1:00)
3. Qualitative predictors and interactions (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
boston = load_dataset('boston')
carseats = load_dataset('carseats')
print('boston   :', boston.shape)
print('carseats :', carseats.shape)


## 1. Multiple LR and collinearity

### Fit `medv` on all 12 predictors

In [ ]:
X = boston.drop(columns=['medv'])
y = boston['medv']
m = LinearRegression().fit(X, y)
coefs = pd.Series(m.coef_, index=X.columns).round(4)
print(coefs.to_string())
print(f'R^2 = {m.score(X, y):.4f}')


### Variance Inflation Factor — VIF

For predictor j, fit `x_j ~ rest`, get R²_j, then VIF_j = 1/(1 - R²_j).
Anything above 5–10 is collinear with the rest of the design.

In [ ]:
def vif(X: pd.DataFrame) -> pd.Series:
    out = {}
    for col in X.columns:
        others = X.drop(columns=[col])
        r2 = LinearRegression().fit(others, X[col]).score(others, X[col])
        out[col] = 1.0 / (1.0 - r2)
    return pd.Series(out).sort_values(ascending=False)

print(vif(X).round(2).to_string())


`tax`, `rad`, `nox` are the biggest offenders. Coefficient *signs* on a
regression with collinear predictors can flip when you add or drop a
feature — never read coefficients in isolation.

## 2. Diagnostic plots and leverage

In [ ]:
y_hat = m.predict(X)
resid = y - y_hat
std_resid = resid / resid.std()

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0, 0].scatter(y_hat, resid, s=8, alpha=0.6)
axes[0, 0].axhline(0, color='C3'); axes[0, 0].set_title('Residuals vs Fitted')
axes[0, 0].set_xlabel('fitted'); axes[0, 0].set_ylabel('residual')

from scipy import stats
stats.probplot(resid, plot=axes[0, 1])
axes[0, 1].set_title('Q-Q')

axes[1, 0].scatter(y_hat, np.sqrt(np.abs(std_resid)), s=8, alpha=0.6)
axes[1, 0].set_title('Scale-Location'); axes[1, 0].set_xlabel('fitted')
axes[1, 0].set_ylabel(r'$\sqrt{|standardized\ resid|}$')

# Leverage: diag of the hat matrix H = X(X'X)^{-1} X'
X_aug = np.column_stack([np.ones(len(X)), X.to_numpy()])
H_diag = np.einsum('ij,ji->i', X_aug @ np.linalg.inv(X_aug.T @ X_aug), X_aug.T)
axes[1, 1].scatter(H_diag, std_resid, s=8, alpha=0.6)
axes[1, 1].axhline(0, color='C3')
axes[1, 1].set_title('Residuals vs Leverage')
axes[1, 1].set_xlabel('leverage'); axes[1, 1].set_ylabel('std residual')
plt.tight_layout(); plt.show()


**Cook's distance** roughly combines leverage and residual size. Any
point in the upper-right of the leverage plot deserves a look.

## 3. Qualitative predictors and interactions

### Carseats: one-hot the categorical predictors

In [ ]:
print(carseats.dtypes)
print(carseats[['ShelveLoc', 'Urban', 'US']].head())


In [ ]:
df = pd.get_dummies(carseats, columns=['ShelveLoc', 'Urban', 'US'],
                    drop_first=True, dtype=float)
X = df.drop(columns=['Sales'])
y = df['Sales']
m = LinearRegression().fit(X, y)
print(pd.Series(m.coef_, index=X.columns).round(3).to_string())
print(f'R^2 = {m.score(X, y):.4f}')


### Add an interaction: Price × Urban

In [ ]:
df['Price_x_Urban'] = df['Price'] * df['Urban_Yes']
X2 = df.drop(columns=['Sales'])
m2 = LinearRegression().fit(X2, y)
print('coef on Price:           ', m2.coef_[X2.columns.get_loc('Price')].round(4))
print('coef on Price_x_Urban:   ', m2.coef_[X2.columns.get_loc('Price_x_Urban')].round(4))
print(f'R^2 with interaction:    {m2.score(X2, y):.4f}')


### Recap
- Multiple regression in matrix form: $\hat\beta = (X^\top X)^{-1} X^\top y$.
- Always check VIF — collinearity inflates standard errors and flips signs.
- The 2×2 diagnostic panel is your first reflex when a model 'looks fine'.
- One-hot encoding turns categoricals into numeric. Interactions are just
  products of two predictors.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Load `penguins`, drop NaN, one-hot `species` and `sex`,
and fit `body_mass_g ~ all other numeric + dummies`. Print R².

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.linear_model import LinearRegression
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.linear_model import LinearRegression
df = load_dataset('penguins').dropna()
df = pd.get_dummies(df, columns=['species', 'sex', 'island'], drop_first=True, dtype=float)
X = df.drop(columns=['body_mass_g'])
y = df['body_mass_g']
m = LinearRegression().fit(X, y)
print(f'R^2 = {m.score(X, y):.4f}')


---

## Exercise 2

**Task 2.** Compute VIF for the numeric predictors (`bill_length_mm`,
`bill_depth_mm`, `flipper_length_mm`). Any of them above 5?

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
numeric = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
Xn = df[numeric]
for col in numeric:
    others = Xn.drop(columns=[col])
    r2 = LinearRegression().fit(others, Xn[col]).score(others, Xn[col])
    print(f'{col:22s}  VIF = {1/(1-r2):.2f}')


---

## Exercise 3

**Task 3.** Add an interaction `flipper_length_mm × species_Chinstrap`
to the model. Does R² improve?

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
df['flipper_x_chinstrap'] = df['flipper_length_mm'] * df['species_Chinstrap']
X2 = df.drop(columns=['body_mass_g'])
m2 = LinearRegression().fit(X2, df['body_mass_g'])
print(f'R^2 with interaction = {m2.score(X2, df["body_mass_g"]):.4f}')
